In [ ]:
from functools import partial

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import torch

import torchlensmaker as tlm

dtype, device = torch.float32, torch.device("cpu")


def plot_sdf(func, tau, lim, title):
    fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(15, 5))

    xspace = torch.linspace(-lim, lim, 300)
    yspace = torch.linspace(-lim, lim, 300)

    X, Y = torch.meshgrid(xspace, yspace, indexing="ij")

    points = torch.stack((X, Y), dim=-1)

    F = func(points, order=1)

    norm = colors.SymLogNorm(
        linthresh=0.05, linscale=1.0, vmin=-20.0, vmax=20.0, base=10
    )
    ax0.pcolormesh(X, Y, F.val, cmap="seismic", norm=norm, shading="gouraud")
    ax0.set_title("F")
    if tau:
        ax0.axhline(y=tau)
        ax0.axhline(y=-tau)
    ax0.set_aspect("equal")

    ax1.pcolormesh(X, Y, F.grad[..., 0])
    ax1.set_title("grad_x")
    ax2.pcolormesh(X, Y, F.grad[..., 1])
    ax2.set_title("grad_r")
    ax1.set_aspect("equal")
    ax2.set_aspect("equal")
    fig.suptitle(f"{title} tau={tau}")


def show_sdf(
    sag_function,
    tau,
    normalize,
    title,
):
    nf = torch.tensor(1 if not normalize else tau, dtype=dtype, device=device)
    tau = torch.as_tensor(tau, dtype=dtype, device=device)
    implicit_function = tlm.sag_to_implicit_2d_raw(sag_function, nf=nf, tau=tau)
    plot_sdf(implicit_function, tau, 1.8 * tau, title)

In [ ]:
show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 1.2)),
    tau=1,
    normalize=False,
    title="spherical 1/10",
)

In [ ]:
import torchlensmaker.implicit as tlmi

plot_sdf(partial(tlmi.implicit_disk_2d, R=1.0), None, 3.0, "disk_2d")
plot_sdf(partial(tlmi.implicit_yzcircle_2d, R=1.0), None, 3.0, "circle_2d")
plot_sdf(tlmi.implicit_yaxis_2d, None, 3.0, "yaxis_2d")

In [ ]:
show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 10)),
    tau=1,
    normalize=False,
    title="spherical 1/10",
)

show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 20)),
    tau=10,
    normalize=False,
    title="spherical 1/20"
)

show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 20)),
    tau=20,
    normalize=False,
    title="spherical 1/20"
)

show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 1)),
    tau=20,
    normalize=True,
    title="spherical 1/1 normalized"
)


show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 10)),
    tau=1,
    normalize=True,
    title="spherical 1/10 normalized"
)

show_sdf(
    partial(tlm.spherical_sag_2d, C=torch.tensor(1 / 2)),
    tau=10,
    normalize=True,
    title="spherical 1/2 normalized"
)

show_sdf(
    partial(
        tlm.aspheric_sag_2d,
        coefficients=torch.distributions.uniform.Uniform(-1.0, 1.0).sample((3,)),
    ),
    tau=1,
    normalize=False,
    title="aspheric"
)

show_sdf(
    partial(
        tlm.conical_sag_2d,
        C=torch.tensor(1 / 15.0, dtype=dtype, device=device),
        K=torch.tensor(0.0, dtype=dtype, device=device),
    ),
    tau=1,
    normalize=False,
    title="conical"
)